# 🦖 ANTIGRAVITY COLAB WATCHER v1.1
### **GPU POWER FOR STEM SEPARATION**

1. **Mount Drive** (Connect to your account)
2. **Run All**
3. **Forget it** - It will watch your Drive for new songs indefinitely!

In [ ]:
# 1. Install Demucs
!pip install -U demucs

# 2. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import subprocess
import shutil

INPUT_DIR = '/content/drive/MyDrive/COLAB_INPUT'
OUTPUT_DIR = '/content/drive/MyDrive/COLAB_OUTPUT'

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("🦖 watcher is online! Waiting for songs in root/COLAB_INPUT...")

while True:
    songs = [f for f in os.listdir(INPUT_DIR) if f.endswith(('.mp3', '.wav'))]
    
    if not songs:
        time.sleep(5)
        continue
        
    for song in songs:
        song_id = song.split('_trimmed')[0]
        print(f"🔥 Processing: {song}")
        
        input_path = os.path.join(INPUT_DIR, song)
        
        # Run Demucs using the GPU
        subprocess.run([
            "demucs", "--two-stems", "vocals", 
            "-o", "/content/stems", 
            "--device", "cuda",
            input_path
        ])
        
        # Find the output folder
        track_name = os.path.splitext(song)[0]
        # The model is 'htdemucs' by default
        results_dir = os.path.join("/content/stems/htdemucs", track_name)
        
        if os.path.exists(results_dir):
            out_folder = os.path.join(OUTPUT_DIR, song_id)
            os.makedirs(out_folder, exist_ok=True)
            
            for f in os.listdir(results_dir):
                shutil.move(os.path.join(results_dir, f), os.path.join(out_folder, f))
            
            print(f"✅ Done: {song_id}")
            os.remove(input_path)
        else:
            print(f"❌ ERROR: Could not find output in {results_dir}")
            os.makedirs('/content/drive/MyDrive/ERRORS', exist_ok=True)
            shutil.move(input_path, os.path.join('/content/drive/MyDrive/ERRORS', song))
            
    time.sleep(2)